# 05 Silver Condition Clean

## Purpose

This notebook creates the Silver Condition table from the Bronze raw Condition FHIR table.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.condition_raw`
2. Extract diagnosis and condition fields
3. Flatten disease-related information
4. Create a patient disease history table
5. Save it as `healthcare_catalog.silver.condition_clean`

## Why We Are Doing This

Condition resources contain patient diagnoses and clinical problems.

This table will later support:
- chronic disease analytics
- patient risk scoring
- readmission prediction
- comorbidity analysis
- population health analytics

## Expected Final Output

A clean Silver table:

`healthcare_catalog.silver.condition_clean`

Expected columns:
- condition_id
- patient_id
- encounter_id
- condition_description
- clinical_status
- verification_status
- onset_datetime
- recorded_datetime

## Step 1 — Import PySpark Functions

### What We Are Doing

We are importing PySpark SQL functions.

### Why We Are Doing This

We need Spark functions to extract nested diagnosis fields from FHIR Condition resources.

### Expected Output

Spark functions available for this notebook.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze Condition Table

### What We Are Doing

We are reading the raw Bronze Condition table.

### Why We Are Doing This

The Bronze layer contains raw nested FHIR Condition resources.

### Expected Output

A DataFrame named:

`condition_raw_df`

In [0]:
condition_raw_df = spark.table(
    "healthcare_catalog.bronze.condition_raw"
)

print("Bronze condition_raw table loaded successfully.")

Bronze condition_raw table loaded successfully.


## Step 3 — Inspect Condition Raw Schema

### What We Are Doing

We are printing the schema of the Condition resource.

### Why We Are Doing This

FHIR diagnosis resources are deeply nested.

We need to identify:
- patient references
- encounter references
- diagnosis descriptions
- status fields
- timestamps

### Expected Output

You should see fields such as:
- resource.id
- resource.subject.reference
- resource.encounter.reference
- resource.code
- resource.clinicalStatus
- resource.verificationStatus
- resource.onsetDateTime
- resource.recordedDate

In [0]:
condition_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean Condition Columns

### What We Are Doing

We are extracting diagnosis-related fields from nested FHIR Condition resources.

### Why We Are Doing This

FHIR Condition resources contain disease and diagnosis information.

Analytics and ML models require flattened diagnosis tables.

This table will become the foundation for:
- chronic disease analytics
- comorbidity analysis
- patient risk scoring
- readmission prediction

### Fields We Will Extract

- condition_id
- patient_reference
- encounter_reference
- condition_description
- clinical_status
- verification_status
- onset_datetime
- recorded_datetime

### Expected Output

A flattened DataFrame named:

`condition_clean_df`

In [0]:
condition_clean_df = condition_raw_df.select(

    col("resource.id").alias("condition_id"),

    col("resource.subject.reference").alias("patient_reference"),

    col("resource.encounter.reference").alias("encounter_reference"),

    get_json_object(col("resource.code"), "$.text").alias("condition_description"),

    col("resource.clinicalStatus.coding")[0]["code"].alias("clinical_status"),

    col("resource.verificationStatus.coding")[0]["code"].alias("verification_status"),

    col("resource.onsetDateTime").alias("onset_datetime"),

    col("resource.recordedDate").alias("recorded_datetime")
)

print("Condition clean DataFrame created successfully.")

Condition clean DataFrame created successfully.


## Step 5 — Convert Condition Timestamps

### What We Are Doing

We are converting diagnosis timestamps into Spark timestamp format.

### Why We Are Doing This

Time-based clinical analytics require timestamp data types.

This supports:
- disease progression analysis
- chronic condition tracking
- longitudinal patient history

### Expected Output

Timestamp columns converted successfully.

In [0]:
condition_clean_df = condition_clean_df.withColumn(
    "onset_datetime",
    to_timestamp(col("onset_datetime"))
)

condition_clean_df = condition_clean_df.withColumn(
    "recorded_datetime",
    to_timestamp(col("recorded_datetime"))
)

print("Condition timestamps converted successfully.")

Condition timestamps converted successfully.


## Step 6 — Extract Clean Patient and Encounter IDs

### What We Are Doing

We are extracting clean UUIDs from FHIR reference fields.

### Why We Are Doing This

FHIR references contain:
- urn:uuid:
- ResourceType/ID

We need clean IDs for table joins.

### Expected Output

New columns:
- patient_id
- encounter_id

In [0]:
condition_clean_df = condition_clean_df.withColumn(
    "patient_id",

    regexp_extract(
        col("patient_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

condition_clean_df = condition_clean_df.withColumn(
    "encounter_id",

    regexp_extract(
        col("encounter_reference"),
        r'urn:uuid:(.*)',
        1
    )
)

print("Patient and encounter IDs extracted successfully.")

Patient and encounter IDs extracted successfully.


## Step 7 — Inspect Clean Condition Data

### What We Are Doing

We are displaying the flattened diagnosis table.

### Why We Are Doing This

We need to verify:
- diagnosis extraction works
- timestamps converted correctly
- patient and encounter IDs extracted correctly

### Expected Output

A clean patient diagnosis history table.

In [0]:
display(condition_clean_df)

condition_id,patient_reference,encounter_reference,condition_description,clinical_status,verification_status,onset_datetime,recorded_datetime,patient_id,encounter_id
efab992a-6a25-872d-9979-b78d7e019505,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:fd473694-aa12-3ab6-2d9b-63569261d9d9,Received certificate of high school equivalency (finding),active,confirmed,2002-08-03T14:54:49.000Z,2002-08-03T14:54:49.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,fd473694-aa12-3ab6-2d9b-63569261d9d9
aca6a067-cb36-4a13-9dd5-daa62cf5fd05,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:1581b40f-37c4-16cd-19eb-8bc37acff711,Full-time employment (finding),resolved,confirmed,2003-08-09T15:02:30.000Z,2003-08-09T15:02:30.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,1581b40f-37c4-16cd-19eb-8bc37acff711
0e9a6e40-29db-c529-d66c-daefdaa1758f,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:1581b40f-37c4-16cd-19eb-8bc37acff711,Limited social contact (finding),resolved,confirmed,2003-08-09T15:02:30.000Z,2003-08-09T15:02:30.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,1581b40f-37c4-16cd-19eb-8bc37acff711
c4d2ff2e-76dd-b158-a1d5-73ddb86698df,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:1581b40f-37c4-16cd-19eb-8bc37acff711,Stress (finding),resolved,confirmed,2003-08-09T15:02:30.000Z,2003-08-09T15:02:30.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,1581b40f-37c4-16cd-19eb-8bc37acff711
3c921e4b-fa06-8e37-c430-a0606a71de96,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:ce24a593-9ad3-8847-0d2d-42ea01ea0f6e,Full-time employment (finding),resolved,confirmed,2006-08-12T15:04:09.000Z,2006-08-12T15:04:09.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,ce24a593-9ad3-8847-0d2d-42ea01ea0f6e
c589fefb-6aad-fdf0-c1be-c16a40824850,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:3cc5adba-4acc-baea-bfcf-67d3c95cfccb,Part-time employment (finding),resolved,confirmed,2009-08-15T15:03:51.000Z,2009-08-15T15:03:51.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,3cc5adba-4acc-baea-bfcf-67d3c95cfccb
28403f1e-24f3-7d1a-09ea-1e936ec18726,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:3cc5adba-4acc-baea-bfcf-67d3c95cfccb,Stress (finding),resolved,confirmed,2009-08-15T15:03:51.000Z,2009-08-15T15:03:51.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,3cc5adba-4acc-baea-bfcf-67d3c95cfccb
1cfba0cb-dc2f-3419-4128-bbe26fadfb5e,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:df5e08d1-ffe5-8ad2-29aa-1b0e096ecff7,Full-time employment (finding),resolved,confirmed,2012-08-18T14:56:57.000Z,2012-08-18T14:56:57.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,df5e08d1-ffe5-8ad2-29aa-1b0e096ecff7
e6efd1f3-5afc-083c-68ac-a1087938b365,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:df5e08d1-ffe5-8ad2-29aa-1b0e096ecff7,Unhealthy alcohol drinking behavior (finding),active,confirmed,2012-08-18T16:12:13.000Z,2012-08-18T16:12:13.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,df5e08d1-ffe5-8ad2-29aa-1b0e096ecff7
a1c3c54f-c2f2-4006-3285-e543ff63ed0d,urn:uuid:58a3b04c-07b8-fd55-b008-316dbd1fe190,urn:uuid:35c08a23-8e16-f879-bc4b-f48c66d0cf00,Normal pregnancy,resolved,confirmed,2013-02-23T14:05:02.000Z,2013-02-23T14:05:02.000Z,58a3b04c-07b8-fd55-b008-316dbd1fe190,35c08a23-8e16-f879-bc4b-f48c66d0cf00


## Step 8 — Check Most Common Conditions

### What We Are Doing

We are counting the most common diagnoses.

### Why We Are Doing This

This provides:
- clinical understanding
- data validation
- early population health analytics

### Expected Output

Top diagnosis frequency table.

In [0]:
display(
    condition_clean_df.groupBy(
        "condition_description"
    ).count().orderBy(
        desc("count")
    )
)

condition_description,count
Full-time employment (finding),6379
Stress (finding),2246
Part-time employment (finding),976
Viral sinusitis (disorder),585
Limited social contact (finding),535
Social isolation (finding),515
Not in labor force (finding),421
Victim of intimate partner abuse (finding),379
Acute viral pharyngitis (disorder),347
Acute bronchitis (disorder),266


## Step 9 — Check Null Values

### What We Are Doing
We are checking missing values in the clean Condition table.

### Why We Are Doing This
Silver tables must be validated before saving and using them in analytics or ML.

### Expected Output
A null-count summary for each column.

In [0]:
display(
    condition_clean_df.select(
        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)
            for column_name in condition_clean_df.columns
        ]
    )
)

condition_id,patient_reference,encounter_reference,condition_description,clinical_status,verification_status,onset_datetime,recorded_datetime,patient_id,encounter_id
0,0,0,0,0,0,0,0,0,0


## Step 10 — Save Silver Condition Table

### What We Are Doing
We are saving the clean diagnosis table into the Silver layer.

### Why We Are Doing This
This creates a reusable Delta table for downstream analytics, dashboards, and ML features.

### Expected Output
A Delta table:

`healthcare_catalog.silver.condition_clean`

In [0]:
condition_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.condition_clean")

print("Silver condition_clean table saved successfully.")

Silver condition_clean table saved successfully.


In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+---------------+-----------+
|database|tableName      |isTemporary|
+--------+---------------+-----------+
|silver  |condition_clean|false      |
|silver  |encounter_clean|false      |
|silver  |patient_clean  |false      |
+--------+---------------+-----------+

